In [ ]:
import os
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import torch
# torch.cuda.init()
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")

torch.cuda.set_device(0)
import json

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



3


In [2]:
from pydantic import BaseModel, Field
from typing import Optional, Union
from datetime import datetime
from tqdm import tqdm

In [3]:
!nvidia-smi

Mon Oct 27 16:27:48 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   32C    P8    21W / 230W |      5MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

In [4]:

!kill -9 3772414

/bin/bash: line 0: kill: (3772414) - No such process


In [5]:
# ---------- Step 1: Load and Process the JSON Records ----------
# # Load JSON records from file
# with open('filtered_records_subset_100.json', 'r') as file:
#     filtered_records = json.load(file)

In [6]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
# documents = []
# metadata = []

In [7]:
# Process records into documents
# documents = []
# metadata = []

# for record in filtered_records:
#     doi = record.get("doi", "Unknown DOI")
    
#     # Add abstract as a separate document
#     abstract_text = record.get("abstract", "").strip()
#     if abstract_text:
#         combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi, "source": "abstract"})
    
#     # Add each paragraph as a separate document
#     for para in record.get("paragraphs", []):
#         paragraph_text = para.get("text", "").strip()
#         if paragraph_text:
#             combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
#             documents.append(combined_text)
#             metadata.append({"doi": doi, "source": "paragraph"})

# print(f"Total documents processed: {len(documents)}")

In [8]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [9]:
# ---------- Step 2: Prepare Embeddings for the Existing FAISS Index ----------
# Use the same model configuration that was used when the index was created.
# embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_3775722/2006348050.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


In [10]:
# ---------- Step 2: Load the Existing Vector Database from Disk ----------
INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"
if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

# Use allow_dangerous_deserialization=True to load indices saved with pickle-based metadata.
vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector database from {INDEX_DIRECTORY} with {vector_db.index.ntotal} vectors")

Loaded vector database from faiss_index with 29300 vectors


In [11]:
# # Print DOIs for up to the first 50 processed documents
# if not metadata:
#     print("No metadata available. Did you load the records?")
# else:
#     for entry in metadata[:50]:
#         print(entry.get("doi", "Unknown DOI"))

In [12]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
# query = "How is hierarchical ZSM-5 synthesized?"
# query = "What is the best way to synthesize ZSM-5 that is hierarchical?"
# query = "How is Silicalite-1 synthesized?"
query = "How can you study the effect of nature of silica source on the purity of template-free ZSM-5?"


In [13]:
# Retrieve top k relevant documents (papers) from the vector database.
retrieved_docs = vector_db.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [14]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [15]:
context_text

'Synthesis of ZSM-5 from template-free batches which preceded the preparation of template-free ZSM-5 layers on porous supports was studied to ascertain the effect of nature of silica source on the purity of template-free ZSM-5. Silicic acid and two colloidal silica sols were used as silica sources to prepare the template-free batches with a molar composition of 6.5Na2O:Al2O3:80SiO2:3196H2O. One of the colloidal silica sols contained methanol as stabilizer while the other did not. The product purity and rate of crystallization increased when colloidal silica sols were used as silica source, however, use of silicic acid led to low purity and slow crystallization rate. The methanol in the colloidal silica sol appeared to act as template to promote the crystallization and was occluded in the resultant ZSM-5 pores. The dissolution of the meta-stable ZSM-5 phase and formation of quartz was observed regardless of the nature of the silica source in case of prolonging the crystallization time m

In [16]:
# RAG prompt template
RAG_PROMPT = """
Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""



In [17]:
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
prompt = rag_prompt.format(context=context_text, question=query)

In [18]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/tokenization_auto.py:655: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("Supported model types:", list(CONFIG_MAPPING.keys()))

Supported model types: ['albert', 'align', 'altclip', 'audio-spectrogram-transformer', 'autoformer', 'bark', 'bart', 'beit', 'bert', 'bert-generation', 'big_bird', 'bigbird_pegasus', 'biogpt', 'bit', 'blenderbot', 'blenderbot-small', 'blip', 'blip-2', 'bloom', 'bridgetower', 'bros', 'camembert', 'canine', 'chinese_clip', 'clap', 'clip', 'clipseg', 'code_llama', 'codegen', 'conditional_detr', 'convbert', 'convnext', 'convnextv2', 'cpmant', 'ctrl', 'cvt', 'data2vec-audio', 'data2vec-text', 'data2vec-vision', 'deberta', 'deberta-v2', 'decision_transformer', 'deformable_detr', 'deit', 'deta', 'detr', 'dinat', 'dinov2', 'distilbert', 'donut-swin', 'dpr', 'dpt', 'efficientformer', 'efficientnet', 'electra', 'encodec', 'encoder-decoder', 'ernie', 'ernie_m', 'esm', 'falcon', 'flaubert', 'flava', 'fnet', 'focalnet', 'fsmt', 'funnel', 'git', 'glpn', 'gpt-sw3', 'gpt2', 'gpt_bigcode', 'gpt_neo', 'gpt_neox', 'gpt_neox_japanese', 'gptj', 'gptsan-japanese', 'graphormer', 'groupvit', 'hubert', 'ibert'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [22]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=10000)

In [23]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)



/tmp/ipykernel_3775722/4067646612.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [24]:
# Generate the answer based on the prompt that includes retrieved context.
# response_text = rag_llm(prompt)
# print("Response:")
# print(response_text)

In [25]:
from pathlib import Path

zeolite_codes = [
    "*-ITN", "*-SVY", "*BEA", "*BEA/BEC", "*CTH", "*MRE", "*SFV", "*STO", "*UOE", "-CLO",
    "-IFT", "-IFU", "-IRY", "-ITV", "-LIT", "-SVR", "ABW", "ACO", "AEI", "AEI/CHA", "AEL",
    "AEN", "AFI", "AFN", "AFO", "AFR", "AFS", "AFV", "AFX", "AFX/CHA", "AFY", "AHT", "ANA",
    "APC", "APD", "AST", "ASU-12", "ASU-14", "ASU-16", "ASV", "ATN", "ATO", "ATS", "ATT", "ATV",
    "AVE", "AVL", "AWO", "AWW", "BCT", "BEC", "BOG", "BPH", "BSV", "CAN", "CAS", "CDO", "CFI",
    "CGF", "CGS", "CHA", "CHA/AEI", "CON", "CSV", "CZP", "DDR", "DFO", "DFT", "DOH", "DON",
    "EAB", "EDI", "EEI", "EMT", "EON", "ERI", "ERI/OFF", "ESV", "ETL", "ETR", "ETV", "EUO",
    "EUO/*MRE", "EWS", "EZT", "FAU", "FAU/EMT", "FDU-4", "FER", "GIS", "GME", "GON", "HEU",
    "IFO", "IFR", "IFW", "IHW", "IM-14", "IMF", "IRN", "IRR", "ISV", "ISV/BEC", "ITE", "ITG",
    "ITH", "ITQ-21", "ITQ-43", "ITR", "ITT", "ITW", "IWR", "IWS", "IWV", "IWW", "JBW", "JRY",
    "JSR", "JST", "JSW", "KFI", "LAU", "LEV", "LOS", "LTA", "LTF", "LTJ", "LTL", "LTN", "MAZ",
    "MEI", "MEL", "MEL/MFI", "MEL/ZSM-55", "MER", "MFI", "MFI/MEL", "MFS", "MOR", "MOR/MFI",
    "MOZ", "MRT", "MSE", "MSO", "MTF", "MTN", "MTT", "MTW", "MVY", "MWF", "MWW", "NAT", "NES",
    "NON", "NSI", "NUD-1", "OFF", "OWE", "PAU", "PHI", "PKU-17", "PON", "POR", "POS", "PTY",
    "PUN", "PWN", "PWO", "PWW", "RHO", "RRO", "RSN", "RTE", "RTH", "RUT", "RUT/RTH", "RWY",
    "SAF", "SAO", "SAS", "SAT", "SAV", "SBE", "SBN", "SBT", "SFE", "SFF", "SFG", "SFH", "SFN",
    "SFO", "SFS", "SFW", "SGT", "SOD", "SOF", "SOR", "SOS", "SOV", "SSF", "SSY", "STF",
    "STF/SFF", "STI", "STT", "STW", "SU-65", "SU-67", "SU-74", "SU-77", "SU-79", "SU-M",
    "SU-MB", "SVV", "SWY", "SYSU-3", "SZR", "THO", "TON", "TON/MTT", "TUN", "UFI", "UOS",
    "UOV", "UOZ", "USI", "UTL", "UWY", "VET", "VFI", "VSV", "YFI", "ZON"
]

questions = [f"Give me synthesis conditions for a {code} recipe." for code in zeolite_codes]

output_path = Path("zeolite_synthesis_questions.txt")
output_path.write_text("\n".join(questions), encoding="utf-8")
print(f"Saved {len(questions)} questions to {output_path.resolve()}")

Saved 233 questions to /home/jupyter/Mrigi/Zeolite_RAG/zeolite_synthesis_questions.txt


In [28]:
# def run_queries_with_rag_llm(queries, codes, rag_chain, output_path, batch_size=2):
#     """Run the provided queries through rag_llm, saving results incrementally."""
#     responses = []
#     output_path = Path(output_path)
#     for start in range(0, len(queries), batch_size):
#         batch_queries = queries[start:start + batch_size]
#         batch_codes = codes[start:start + batch_size]
#         for code, query_text in zip(batch_codes, batch_queries):
#             try:
#                 response = rag_chain.invoke(query_text)
#             except Exception as exc:
#                 response = f"Error: {exc}"
#             record = {
#                 "framework": code,
#                 "query": query_text,
#                 "response": str(response)
#             }
#             responses.append(record)
#             output_path.write_text(json.dumps(responses, indent=2), encoding="utf-8")
#     return responses

# output_json = Path("zeolite_synthesis_responses.json")
# rag_responses = run_queries_with_rag_llm(questions, zeolite_codes, rag_llm, output_json)
# print(f"Saved responses for {len(rag_responses)} frameworks to {output_json.resolve()}")

def run_queries_with_rag_llm(queries, codes, rag_chain, output_path, batch_size=2):
    """Run the provided queries through rag_llm, saving results incrementally."""
    responses = []
    output_path = Path(output_path)
    
    for start in tqdm(range(0, len(queries), batch_size)):
        batch_queries = queries[start:start + batch_size]
        batch_codes = codes[start:start + batch_size]
        
        for code, query_text in zip(batch_codes, batch_queries):
            try:
                # Get relevant documents first
                retrieved_docs = vector_db.similarity_search(query_text, k=4)
                context = "\n\n".join([doc.page_content for doc in retrieved_docs])
                
                # Format the prompt properly
                prompt = rag_prompt.format(context=context, question=query_text)
                
                # Get response using the formatted prompt
                response = rag_llm(prompt)
                
            except Exception as exc:
                response = f"Error: {exc}"
                
            record = {
                "framework": code,
                "query": query_text,
                "response": str(response)
            }
            responses.append(record)
            
            # Save after each response
            with output_path.open('w', encoding='utf-8') as f:
                json.dump(responses, f, indent=2)
                
    return responses


output_json = Path("zeolite_synthesis_responses.json")
rag_responses = run_queries_with_rag_llm(questions, zeolite_codes, rag_llm, output_json)
print(f"Saved responses for {len(rag_responses)} frameworks to {output_json.resolve()}")


  0%|          | 0/117 [00:00<?, ?it/s]/tmp/ipykernel_3775722/3432764362.py:45: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag_llm(prompt)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/tmp/ipykernel_3775722/3432764362.py:45: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag_llm(prompt)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  1%|          | 1/117 [00:06<12:21,  6.39s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end 

Saved responses for 233 frameworks to /home/jupyter/Mrigi/Zeolite_RAG/zeolite_synthesis_responses.json
